Exactly same as v5 but uses reasoning ("ChainOfTHought") for the second DSPy agent (the one that finds relevant datasets from top-k matched datasets. It also changed the value of `k` from `5`to `10`to get the top `10`closest datasets to our user question embeddings based on similarity score (cosine similiarity)

In [ ]:
!pip -q install requests psycopg[binary] pgvector openai azure-core dspy-ai tiktoken

In [ ]:
import os
from google.colab import userdata

# Azure OpenAI
os.environ["AZURE_OPENAI_ENDPOINT"] = "https://o3miniapi.openai.azure.com/"
os.environ["AZURE_OPENAI_API_KEY"] = userdata.get('Azure')
os.environ["AZURE_OPENAI_EMBED_DEPLOYMENT"] = "text-embedding-3-small"
os.environ["AZURE_OPENAI_CHAT_DEPLOYMENT"] = "gpt-5-mini"

# Neon (optional for now)
os.environ["NEON_DATABASE_URL"] = userdata.get("Neon")  # put your Neon URL in Colab secrets



Next code block is for functions to fill out the NEON DB with chosen dataset embeddings and also function to search for similar datasets given a user quesion (using cosine similarity)

In [ ]:

from __future__ import annotations

import os
import time
import json
from dataclasses import dataclass
from typing import Any, Dict, Iterable, List, Optional, Sequence, Set, Tuple
import re
import requests

from openai import AzureOpenAI

# Postgres / pgvector
# pip install psycopg[binary] pgvector
import psycopg
from pgvector.psycopg import register_vector
import tiktoken


# ----------------------------
# 1) data.gouv.fr API fetching
# ----------------------------

DATASETS_URL = "https://www.data.gouv.fr/api/1/datasets/"

# openai.api_type = "azure"
# openai.api_base = os.environ["AZURE_OPENAI_ENDPOINT"]
# openai.api_key = os.environ["AZURE_OPENAI_API_KEY"]
# openai.api_version = "2024-02-01"

_URL_RE = re.compile(r"https?://|www\.", re.IGNORECASE)

def _collapse_ws(s: str) -> str:
    return " ".join(s.split())

# OpenAI / Azure embedding models use this encoding
_TIKTOKEN_ENC = tiktoken.get_encoding("cl100k_base")

MAX_DESC_TOKENS = 7000   # hard cap for DESCRIPTION only

def truncate_desc_tokens(desc: str, max_tokens: int = MAX_DESC_TOKENS) -> str:
    tokens = _TIKTOKEN_ENC.encode(desc)

    if len(tokens) <= max_tokens:
        return desc

    truncated = _TIKTOKEN_ENC.decode(tokens[:max_tokens])
    return truncated + " …"

@dataclass
class DatasetRecord:
    id: str
    title: str
    description: str
    tags: List[str]
    organization: Optional[str] = None  # org display name


    @staticmethod
    def clean_tags( # not used. Potentially added in the future.
                   # the problem with tags:
                    # they often include noisy information that
                    # might throw off the embedder
                    # such as data resource types (CSV, JSON, etc)
        tags: Optional[Iterable[str]],
        max_tags: int = 30,
        max_tag_len: int = 60,
        min_tag_len: int = 2,
    ) -> List[str]:
        if not tags:
            return []

        out: List[str] = []
        seen = set()

        for t in tags:
            if not isinstance(t, str):
                continue
            t = _collapse_ws(t.strip())
            if not t:
                continue

            # drop URLs / obvious junk
            if _URL_RE.search(t):
                continue

            # drop super-long tags (often pasted phrases)
            if len(t) > max_tag_len:
                continue

            if len(t) < min_tag_len:
                continue

            key = t.casefold()
            if key in seen:
                continue
            seen.add(key)

            out.append(t)
            if len(out) >= max_tags:
                break

        return out

    def to_embedding_text(self) -> str:
        """
        Stable, readable embedding text.
        - Title & organization are never truncated
        - Description is capped to ~7000 tokens (token-aware)
        """
        title = (self.title or "").strip()
        desc = (self.description or "").strip()
        org = (self.organization or "").strip()

        if desc:
            tokens = _TIKTOKEN_ENC.encode(desc)
            if len(tokens) > MAX_DESC_TOKENS:
                print(
                    f"[desc-truncated] "
                    f"dataset={self.id} "
                    f"tokens={len(tokens)} → {MAX_DESC_TOKENS}"
                )
                desc = _TIKTOKEN_ENC.decode(tokens[:MAX_DESC_TOKENS]) + " …"

        parts = ["DATASET"]

        if title:
            parts.append(f"TITLE: {title}")
        if desc:
            parts.append(f"DESCRIPTION: {desc}")
        if org:
            parts.append(f"ORGANIZATION: {org}")

        return "\n".join(parts).strip()


class DataGouvDatasetsClient:
    def __init__(self, base_url: str = DATASETS_URL, timeout_s: int = 60):
        self.base_url = base_url
        self.timeout_s = timeout_s
        self.session = requests.Session()
        self.session.headers.update({"Accept": "application/json"})

    def fetch_page(
        self,
        page: int = 1,
        page_size: int = 50,
        q: Optional[str] = None,
        tags: Optional[Sequence[str]] = None,  # NEW
    ) -> Dict[str, Any]:
        """
        GET /api/1/datasets/?page=...&page_size=...&q=...&tag=...&tag=...

        NOTE: 'tag' is documented as a string[] query param, meaning it can be repeated.
        """
        params: Dict[str, Any] = {"page": page, "page_size": page_size}
        if q:
            params["q"] = q
        if tags:
            # requests encodes list values as repeated query params: tag=a&tag=b
            params["tag"] = list(tags)

        resp = self.session.get(self.base_url, params=params, timeout=self.timeout_s)
        resp.raise_for_status()
        return resp.json()

    # A bunch of helper functions for the iter_datasets function....
    @staticmethod
    def _pick_str(x: Any) -> Optional[str]:
        if x is None:
            return None
        if isinstance(x, str):
            s = x.strip()
            return s or None
        return None

    @staticmethod
    def _org_name(item: dict) -> Optional[str]:
        org = item.get("organization")
        if isinstance(org, dict):
            return (org.get("name") or org.get("title") or org.get("slug") or "").strip() or None
        return None

    def iter_datasets(
        self,
        mode: str = "single_page",   # "single_page" or "all_pages"
        page: int = 1,
        page_size: int = 50,
        q: Optional[str] = None,
        tags: Optional[Sequence[str]] = None,      # server-side filter
        hard_limit: Optional[int] = None,
        require_any_of_tags: Optional[Set[str]] = None,  # client-side OR verification
    ) -> Iterable["DatasetRecord"]:

        if mode not in ("single_page", "all_pages"):
            raise ValueError("mode must be 'single_page' or 'all_pages'")

        fetched = 0
        current_page = page
        next_url: Optional[str] = None

        while True:
            if next_url is None:
                payload = self.fetch_page(page=current_page, page_size=page_size, q=q, tags=tags)
            else:
                resp = self.session.get(next_url, timeout=self.timeout_s)
                resp.raise_for_status()
                payload = resp.json()

            data = payload.get("data", []) or []
            for item in data:
                rec = DatasetRecord(
                    id=str(item.get("id", "") or ""),
                    title=str(item.get("title", "") or ""),
                    description=str(item.get("description", "") or ""),
                    tags=list(item.get("tags", []) or []),
                    organization=self._org_name(item),
                )

                if not rec.id and not rec.title and not rec.description and not rec.tags:
                    continue

                if require_any_of_tags:
                    if not set(rec.tags).intersection(require_any_of_tags):
                        continue

                yield rec
                fetched += 1
                if hard_limit is not None and fetched >= hard_limit:
                    return

            if mode == "single_page":
                return

            next_url = payload.get("next_page") or None
            if not next_url:
                return

    def iter_datasets_with_any_tags(
        self,
        any_tags: Sequence[str],          # NEW: OR semantics
        mode: str = "all_pages",
        page: int = 1,
        page_size: int = 50,
        q: Optional[str] = None,
        hard_limit: Optional[int] = None,
    ) -> Iterable["DatasetRecord"]:
        """
        Guaranteed OR semantics across tags by querying once per tag and deduplicating.

        Example:
          any_tags=["tarifs-voyageurs", "gtfs", "mobilite"]
        """
        want: Set[str] = set(any_tags)
        seen_ids: Set[str] = set()
        emitted = 0

        for tag in want:
            for rec in self.iter_datasets(
                mode=mode,
                page=page,
                page_size=page_size,
                q=q,
                tags=[tag],                    # server-side filter per tag
                require_any_of_tags=want,      # safety verification (OR)
                hard_limit=None,               # we enforce hard_limit after dedupe
            ):
                if rec.id in seen_ids:
                    continue
                seen_ids.add(rec.id)
                yield rec
                emitted += 1
                if hard_limit is not None and emitted >= hard_limit:
                    return

    # NEW: fetch the *full* dataset object (this is how we get “ALL info” - this
    # will be used later if we want to get full information about a specific
    # dataset.
    def fetch_dataset_detail(self, dataset_id: str) -> Dict[str, Any]:
        url = f"{self.base_url}{dataset_id}/"
        resp = self.session.get(url, timeout=self.timeout_s)
        resp.raise_for_status()
        return resp.json()

# ----------------------------
# 2) Embeddings (Azure OpenAI)
# ----------------------------


class AzureEmbeddingClient:
    def __init__(
        self,
        azure_endpoint: str,
        api_key: str,
        deployment: str,                 # your Azure *deployment name*
        api_version: str = "2024-02-01", # embeddings API version
        request_batch_size: int = 64,
        max_retries: int = 6,
    ):
        self.deployment = deployment
        self.request_batch_size = request_batch_size
        self.max_retries = max_retries

        # NOTE: azure_endpoint (NOT endpoint)
        self.client = AzureOpenAI(
            azure_endpoint=azure_endpoint,
            api_key=api_key,
            api_version=api_version,
        )

    def embed_texts(self, texts: List[str]) -> List[List[float]]:
        all_embeddings: List[List[float]] = []
        total_tokens = 0
        for start in range(0, len(texts), self.request_batch_size):
            batch = texts[start : start + self.request_batch_size]
            attempt = 0
            while True:
                try:
                    resp = self.client.embeddings.create(
                        model=self.deployment,  # In Azure, model=<deployment_name>
                        input=batch,
                    )
                    total_tokens += resp.usage.total_tokens
                    # Keep order stable
                    items = sorted(resp.data, key=lambda x: x.index)
                    all_embeddings.extend([it.embedding for it in items])
                    break
                except Exception as e:
                    attempt += 1
                    if attempt > self.max_retries:
                        raise
                    sleep_s = min(2 ** attempt, 30)
                    print(f"[embed retry {attempt}/{self.max_retries}] {type(e).__name__}: {e} -> sleeping {sleep_s}s")
                    time.sleep(sleep_s)

        PRICE_PER_1M_TOKENS_USD = 0.02  # for text-embedding-3-small (OpenAI list price)
        cost_usd = total_tokens / 1_000_000 * PRICE_PER_1M_TOKENS_USD
        print("🪲 tokens used for embedding:", total_tokens, "estimated USD:", cost_usd)

        return all_embeddings



# ----------------------------
# 3) Neon Postgres + pgvector
# ----------------------------

class PgvectorStore:
    def __init__(self, postgres_url: str):
        self.postgres_url = postgres_url

    def connect(self):
        conn = psycopg.connect(self.postgres_url)

        # Make sure vector type exists BEFORE register_vector()
        with conn.cursor() as cur:
            cur.execute("CREATE EXTENSION IF NOT EXISTS vector;")
        conn.commit()

        register_vector(conn)
        return conn

    def ensure_schema(self, dim: int, table: str = "dataset_embeddings_micro"):
        create_sql = f"""
        CREATE EXTENSION IF NOT EXISTS vector;

        CREATE TABLE IF NOT EXISTS {table} (
            dataset_id TEXT PRIMARY KEY,
            title TEXT,
            description TEXT,
            organization TEXT,
            content TEXT NOT NULL,
            embedding VECTOR({dim}) NOT NULL,
            updated_at TIMESTAMPTZ NOT NULL DEFAULT NOW()
        );

        CREATE INDEX IF NOT EXISTS {table}_embedding_cosine_idx
          ON {table} USING hnsw (embedding vector_cosine_ops);
        """
        with self.connect() as conn:
            with conn.cursor() as cur:
                cur.execute(create_sql)
            conn.commit()

    def upsert_rows(self, rows, table: str = "dataset_embeddings_micro"):
        upsert_sql = f"""
        INSERT INTO {table} (
            dataset_id, title, description,
            organization,
            content, embedding
        )
        VALUES (%s, %s, %s,
                %s,
                %s, %s)
        ON CONFLICT (dataset_id) DO UPDATE SET
            title = EXCLUDED.title,
            description = EXCLUDED.description,
            organization = EXCLUDED.organization,
            content = EXCLUDED.content,
            embedding = EXCLUDED.embedding,
            updated_at = NOW();
        """
        with self.connect() as conn:
            with conn.cursor() as cur:
                cur.executemany(upsert_sql, rows)
            conn.commit()

    def search( # need to change table name here to search within big dataset
        self,
        query_embedding: List[float],
        k: int = 10,
        table: str = "dataset_embeddings_v4",
    ):
        """
        Returns top-k rows by cosine distance.
        """
        sql = f"""
        SELECT
            dataset_id,
            title,
            description,
            organization,
            content,
            (embedding <=> (%s::vector)) AS distance
        FROM {table}
        ORDER BY (embedding <=> (%s::vector)) ASC
        LIMIT %s;
        """

        with self.connect() as conn:
            with conn.cursor() as cur:
                cur.execute(sql, (query_embedding, query_embedding, k))
                return cur.fetchall()

In [ ]:
# ----------------------------
# 4) Orchestration / main
# ----------------------------

def run_pipeline(
    mode: str = "single_page",   # "single_page" or "all_pages"
    page: int = 1,
    page_size: int = 50,
    q: Optional[str] = None,
    hard_limit: Optional[int] = None,
    any_tags: Optional[Sequence[str]] = None,  # NEW
):
    azure_endpoint = os.environ.get("AZURE_OPENAI_ENDPOINT", "")
    azure_api_key = os.environ.get("AZURE_OPENAI_API_KEY", "")
    azure_deployment = os.environ.get("AZURE_OPENAI_EMBED_DEPLOYMENT", "text-embedding-3-small")
    neon_postgres_url = os.environ.get("NEON_DATABASE_URL", "")

    if not azure_endpoint or not azure_api_key:
        raise RuntimeError("Set AZURE_OPENAI_ENDPOINT and AZURE_OPENAI_API_KEY env vars first.")

    dg = DataGouvDatasetsClient()

    # 1) fetch datasets (NEW: OR-tag mode)
    if any_tags:
        records = list(
            dg.iter_datasets_with_any_tags(
                any_tags=any_tags,
                mode="all_pages",       # OR-tag mode should generally scan all pages
                page=page,
                page_size=page_size,
                q=q,
                hard_limit=hard_limit,
            )
        )
    else:
        records = list(dg.iter_datasets(mode=mode, page=page, page_size=page_size, q=q, hard_limit=hard_limit))

    # 2) build embedding texts
    texts = [r.to_embedding_text() for r in records]

    # 3) embed
    emb_client = AzureEmbeddingClient(
        azure_endpoint=os.environ["AZURE_OPENAI_ENDPOINT"],
        api_key=os.environ["AZURE_OPENAI_API_KEY"],
        deployment=os.environ.get("AZURE_OPENAI_EMBED_DEPLOYMENT", "text-embedding-3-small"),
    )

    embeddings = emb_client.embed_texts(texts)

    if len(embeddings) != len(records):
        raise RuntimeError(f"Embedding count mismatch: {len(embeddings)} != {len(records)}")

    dim = len(embeddings[0]) if embeddings else 0
    print(f"Fetched {len(records)} datasets, embedding_dim={dim}")

    # 4) upsert into pgvector (optional)
    if neon_postgres_url:
        store = PgvectorStore(neon_postgres_url)
        store.ensure_schema(dim=dim, table="dataset_embeddings_micro")

        rows = []
        for rec, content, emb in zip(records, texts, embeddings):
            rows.append((
                rec.id,
                rec.title,
                rec.description,
                rec.organization,
                content,
                emb
            ))
        store.upsert_rows(rows, table="dataset_embeddings_micro")
        print(f"Upserted {len(rows)} rows into Neon.")
    else:
        print("NEON_DATABASE_URL not set yet — skipping DB write.")

    return records, embeddings

Next code block runs the pipeline to:
- create embeddings
- store embeddings in NEON DB

YOU DO NOT NEED TO run the following code block(in fact, you SHOULD NOT run the following code block) ify you just want to find similarity between a user question at the datasets stored in the DB (the code to do that is below this code block)

In [ ]:
records, embeddings = run_pipeline(
    any_tags=["tarifs-voyageurs", "autobus", "vie-etudiante"],
    page_size=50,
    hard_limit=2000,   # optional
)

# records, embeddings = run_pipeline(  # <--- already did this, the dataset name is dataset_embeddings_v4
#     mode="all_pages",
#     page_size=50,
#     hard_limit=20000,   # optional
# )

Next code block defines a function to search for the most relevant datasets given a user question (this will be used by the DSPy agent later).

In [ ]:
def search_datasets(query: str, k: int = 5):
    emb_client = AzureEmbeddingClient(
        azure_endpoint=os.environ["AZURE_OPENAI_ENDPOINT"],
        api_key=os.environ["AZURE_OPENAI_API_KEY"],
        deployment=os.environ["AZURE_OPENAI_EMBED_DEPLOYMENT"],
    )
    q_emb = emb_client.embed_texts([query])[0]

    store = PgvectorStore(os.environ["NEON_DATABASE_URL"])
    return store.search(q_emb, k=k)

In [ ]:
# hits = search_datasets("données de transport en commun GTFS en France", k=5)
# for dataset_id, title, desc, tags, content, dist in hits:
#     print(dist, title, dataset_id)

# This was just a test used to check if searach_datasets() functions properly, which it does.

The next few blocks of code contain the setup and use of a simple DSPy agent that takes a user question, adds more context to it, and uses the search_datasets() function to find the most similar datasets.

In [ ]:
import dspy
from datetime import datetime, timezone
from google.colab import userdata
from __future__ import annotations

AZURE_API_KEY = userdata.get('Azure')          # <-- put your key in Colab secrets or env var
AZURE_API_BASE = "https://o3miniapi.cognitiveservices.azure.com/"  # your endpoint
AZURE_API_VERSION = "2024-12-01-preview"
AZURE_DEPLOYMENT = "gpt-5-mini"       # <-- your chat-capable deployment name (e.g., "gpt-4o-mini")

# DSPy LM via LiteLLM Azure provider string.
# Note: provider strings can vary across LiteLLM versions; "azure/<deployment>" is the common pattern.
lm = dspy.LM(
    f"azure/{AZURE_DEPLOYMENT}",
    api_key=os.getenv("AZURE_API_KEY", ""),
    api_base=AZURE_API_BASE,
    api_version=AZURE_API_VERSION,
    model_type="chat",
    temperature=None,
    max_tokens=None,
)

dspy.configure(lm=lm)

In [ ]:
# for usage information

import logging

# More verbose internal logs (optional but useful)
logging.getLogger("dspy").setLevel(logging.DEBUG)  # DSPy FAQ mentions this approach. :contentReference[oaicite:1]{index=1}

# Track token usage (DSPy 2.6.16+)
dspy.configure(track_usage=True)  # usage can be read from Prediction.get_lm_usage(). :contentReference[oaicite:2]{index=2}

In [ ]:
# -------- Signature --------
class PrepareQuery(dspy.Signature):
    """
    Keep the original question unchanged.
    Produce a French version:
      - If already French: minimal grammar/spelling fixes only.
      - If not French: translate to French.
    Produce 3–5 French keywords.
    """
    original_question: str = dspy.InputField()
    french_question: str = dspy.OutputField()
    keywords: List[str] = dspy.OutputField()


# -------- Agent --------
class DataGouvDatasetSearchAgent(dspy.Module):
    def __init__(self, k: int = 5, timezone: str = "Europe/Paris"):
        super().__init__()
        self.k = k
        self.tz = timezone

        # Predict = no chain-of-thought, no over-reasoning
        self.prepare = dspy.Predict(PrepareQuery)

    def forward(self, question: str) -> dspy.Prediction:
        # 1) Keep question EXACTLY as-is
        original_question = question

        # 2) Translate / lightly correct + extract keywords
        prep = self.prepare(original_question=original_question)
        french_q = (prep.french_question or "").strip()

        # 3) Normalize keywords (3–5, ';' separated)
        kws = []
        for kw in (prep.keywords or []):
            kw = str(kw).strip().strip(";")
            if kw:
                kws.append(kw)
        kws = kws[:5]

        keywords_str = "; ".join(kws)


        # 5) Final embedded text (compact, embedding-friendly)
        embedded_text = "\n".join(
            part for part in [
                original_question,
                french_q,
                keywords_str,
            ]
            if part
        )

        # 6) Vector search (your embedding + Neon cosine similarity)
        hits = search_datasets(embedded_text, k=self.k)

        return dspy.Prediction(
            original_question=original_question,
            french_question=french_q,
            keywords=kws,
            embedded_text=embedded_text,
            k=self.k,
            hits=hits,
        )

In [ ]:
# ---------------------------- Second Agent

class SelectUsefulDatasets(dspy.Signature):
    """
    Decide which datasets (if any) are genuinely useful for answering the user's question.

    Input:
      - user_question: the ORIGINAL user question
      - candidates: a numbered list of datasets (id/title/desc/org)
    Output:
      - selected_dataset_ids: ONLY dataset IDs judged useful; return [] if none are useful.
    """
    user_question: str = dspy.InputField()
    candidates: str = dspy.InputField()
    selected_dataset_ids: List[str] = dspy.OutputField()


class DatasetSelectionAgent(dspy.Module):
    def __init__(self):
        super().__init__()
        self.select = dspy.ChainOfThought(SelectUsefulDatasets)

    def forward(self, user_question: str, candidates: str) -> dspy.Prediction:
        out = self.select(user_question=user_question, candidates=candidates)

        # Normalize / de-dupe while preserving order
        seen = set()
        cleaned: List[str] = []
        for x in (out.selected_dataset_ids or []):
            dsid = str(x).strip()
            if dsid and dsid not in seen:
                seen.add(dsid)
                cleaned.append(dsid)

        return dspy.Prediction(selected_dataset_ids=cleaned)

# ----------------------------
# Helper: build the numbered candidates string
# ----------------------------

def _format_candidates(hits: List[Tuple[Any, Any, Any, Any, Any, Any]]) -> str:
    """
    hits expected shape (per your print loop):
      (dataset_id, title, desc, organization, content, dist)

    We will include dataset_id/title/desc/organization as requested.
    """
    lines: List[str] = []
    for i, row in enumerate(hits or [], start=1):
        # Be defensive about row length
        dataset_id = row[0] if len(row) > 0 else ""
        title      = row[1] if len(row) > 1 else ""
        desc       = row[2] if len(row) > 2 else ""
        org        = row[3] if len(row) > 3 else ""

        lines.append(
            "\n".join([
                f"{i})",
                f"dataset_id: {dataset_id}",
                f"title: {title}",
                f"description: {desc}",
                f"organization: {org}",
            ])
        )
    return "\n\n".join(lines).strip()



In [ ]:
def retrieve_and_select_dataset_ids(user_question: str, k: int = 5) -> List[str]:
    """
    1) Runs your DataGouvDatasetSearchAgent on the user question
    2) Concatenates ALL top-k datasets into a numbered string
    3) Runs a second DSPy agent to pick the useful dataset_ids (or none)
    4) Returns ONLY the selected dataset IDs
    """
    # First agent (exactly as you described)
    agent = DataGouvDatasetSearchAgent(k=k)
    result = agent(question=user_question)

    print("\nEmbedded User Query")
    print(result.embedded_text)
    print("\nTop hits:")
    for dataset_id, title, desc, organization, content, dist in result.hits:
        print(dist, dataset_id, title, organization)

    # Build candidates string from hits
    candidates_str = _format_candidates(result.hits)
    print("\nCandidates:")
    print(candidates_str)

    # Second agent: select useful dataset IDs
    selector = DatasetSelectionAgent()
    selection = selector(user_question=user_question, candidates=candidates_str)

    return selection.selected_dataset_ids

In [ ]:
# 3) Example usage
ids = retrieve_and_select_dataset_ids("what are the vacation days official holidays?", k=10)
print("Selected dataset IDs:")
print(ids)


🪲 tokens used for embedding: 35 estimated USD: 7e-07

Embedded User Query
what are the vacation days official holidays?
Quels sont les jours fériés officiels ?
jours fériés; congés; vacances; jours officiels

Top hits:
0.6237466521187618 674905d2097bcecd53bef7fc Abattements taxe de séjour_Données DELTA 2024 et 2025 Ministères économiques et financiers
0.6358001719653656 674905a9097bcecd53bef7fb Tarifs taxe de séjour - Delta 2024 et  2025 Ministères économiques et financiers
0.6394948363304138 69325a329d547a77b34d2b81 Calendrier scolaire 2025-2026 en Zone C - Académie de Toulouse Conseil départemental de la Haute-Garonne - Haute-Garonne Open Data
0.6491234124751766 6902e2c9857e19cd90baf29e Semaine de l'industrie 2025 Région Île-de-France
0.6505842700774749 67a190eaa46fbf38f5c6a497 Programmes de mobilité enseignante gérés par France Éducation international Ministères de l'Éducation nationale
0.6527100801467896 67adf208fff16d427cc86a5e Calendrier d’ouverture des bureaux de poste, agences 

To test my function against created list of questions:

In [ ]:
def evaluate_dataset_against_question(
    user_question: str,
    dataset_id: str,
    k: int = 5
) -> Dict[str, object]:
    """
    Runs retrieve_and_select_dataset_ids on a user question and evaluates
    whether a specific dataset_id is selected.

    Returns a structured schema describing the result.
    """

    # Run your existing pipeline
    selected_ids: List[str] = retrieve_and_select_dataset_ids(
        user_question=user_question,
        k=k
    )

    found = dataset_id in selected_ids

    return {
        "question": user_question,
        "dataset_id": dataset_id,
        "found_dataset": found,
        "chosen_datasets": selected_ids,
    }

In [ ]:
result = evaluate_dataset_against_question(
    user_question="For a local authority considering a rail subsidy, estimate the potential annual subsidy cost to reduce the maximum second-class fare to €15 for all trips from a defined set of commuter origins (assume X trips per commuter per year).",
    dataset_id="62a1706ee2bc92c77ef3ae25",
    k=10
)

print(f"Question: {result['question']}")
print(f"Dataset ID: {result['dataset_id']}")
print(f"Found dataset: {'✅ Yes' if result['found_dataset'] else '❌ No'}")
print(f"Chosen datasets: {result['chosen_datasets']}")


In [ ]:
print("\nDSPy LLM CALL HISTORY (recent)")
print("=" * 80)
dspy.inspect_history(n=1)


DSPy LLM CALL HISTORY (recent)




[2026-02-16T19:28:31.746912]

System message:

Your input fields are:
1. `user_question` (str): 
2. `candidates` (str):
Your output fields are:
1. `reasoning` (str): 
2. `selected_dataset_ids` (list[str]):
All interactions will be structured in the following way, with the appropriate values filled in.

[[ ## user_question ## ]]
{user_question}

[[ ## candidates ## ]]
{candidates}

[[ ## reasoning ## ]]
{reasoning}

[[ ## selected_dataset_ids ## ]]
{selected_dataset_ids}        # note: the value you produce must adhere to the JSON schema: {"type": "array", "items": {"type": "string"}}

[[ ## completed ## ]]
In adhering to this structure, your objective is: 
        Decide which datasets (if any) are genuinely useful for answering the user's question.
        
        Input:
          - user_question: the ORIGINAL user question
          - candidates: a numbered list of datasets (id/title/desc/org)
        Output:
          - selected_dataset_ids: ONLY

In [ ]:
print("LM history length:", len(lm.history), "   --> (Note: this is probably number of times that the LLM was called IN TOTAL - not just in the last run)")
print("Last call keys:", lm.history[-1].keys())
print("Last call usage:", lm.history[-1].get("usage"))
print("Last call cost:", lm.history[-1].get("cost"))

LM history length: 34    --> (Note: this is probably number of times that the LLM was called IN TOTAL - not just in the last run)
Last call keys: dict_keys(['prompt', 'messages', 'kwargs', 'response', 'outputs', 'usage', 'cost', 'timestamp', 'uuid', 'model', 'response_model', 'model_type'])
Last call usage: {'completion_tokens': 781, 'prompt_tokens': 3349, 'total_tokens': 4130, 'completion_tokens_details': CompletionTokensDetailsWrapper(accepted_prediction_tokens=0, audio_tokens=0, reasoning_tokens=448, rejected_prediction_tokens=0, text_tokens=None, image_tokens=None), 'prompt_tokens_details': PromptTokensDetailsWrapper(audio_tokens=0, cached_tokens=0, text_tokens=None, image_tokens=None)}
Last call cost: 0.00239925


The following code block is just used to visualize the embeddings created earlier...

It only works if you run the code block that looks like:

 records, embeddings = run_pipeline(
    any_tags=["tarifs-voyageurs", "autobus", "vie-etudiante"],
    page_size=50,
    hard_limit=2000,   # optional
)

# records, embeddings = run_pipeline(
#     mode="all_pages",
#     page_size=50,
#     hard_limit=20000,
# )

from earlier...

In [ ]:
# Optional : To visualize the embeddings using PCA

import numpy as np
import matplotlib.pyplot as plt
from sklearn.decomposition import PCA



# embeddings: List[List[float]]
E = np.array(embeddings, dtype=np.float32)

# PCA EMBEDDINGS

print("PCA EMBEDDINGS...\n")
# reduce to 2D
XY = PCA(n_components=2, random_state=0).fit_transform(E)

plt.figure(figsize=(8,6))
plt.scatter(XY[:,0], XY[:,1], s=8)
plt.title("Embeddings (PCA 2D)")
plt.xlabel("PC1")
plt.ylabel("PC2")
plt.show()